# Домашняя работа №2

**ФИО**: Дергалов Никита Олегович

**Группа**: ИУ6-55Б

**Вариант**: №2

## Задание:

- оцените дистанцию поездок (в метрах) на основе координат начальной и конечной станций (без учета поездок, завершившихся там же где и начались).
- выведите максимальное, среднее значение, стандартное отклонение и медиан.

### Подключаем Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, avg, stddev, max as spark_max
from pyspark.sql.types import DoubleType
import math

In [2]:
spark = SparkSession.builder.appName('CitiBikeDistance').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/21 19:49:25 WARN Utils: Your hostname, Kvasik, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface enp7s0)
25/11/21 19:49:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/21 19:49:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Оценим дистанцию поездок (в метрах) на основе координат начальной и конечной станций (без учета поездок, завершившихся там же где и начались).

In [ ]:
# Обернем файл в DataFrame
df = spark.read.csv('202301-citibike-tripdata_1.csv', header=True, inferSchema=True)
df

DataFrame[ride_id: string, rideable_type: string, started_at: timestamp, ended_at: timestamp, start_station_name: string, start_station_id: string, end_station_name: string, end_station_id: string, start_lat: double, start_lng: double, end_lat: double, end_lng: double, member_casual: string]

In [4]:
# Формула гаверсина в виде UDF: https://en.wikipedia.org/wiki/Haversine_formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2*R*math.asin(math.sqrt(a))

haversine_udf = udf(haversine, DoubleType())

In [7]:
# Фильтрация поездок, у которых станции разные
filtered = df.filter(col('start_station_id') != col('end_station_id'))

# Добавление расстояния (в метрах)
filtered = filtered.withColumn('distance_meters', haversine_udf(
    col('start_lat'), col('start_lng'),
    col('end_lat'), col('end_lng')
))

In [ ]:
filtered.head() # поле distance_meters добавлено

Row(ride_id='A8518A6C4BE513DE', rideable_type='classic_bike', started_at=datetime.datetime(2023, 1, 3, 23, 14, 52, 325000), ended_at=datetime.datetime(2023, 1, 3, 23, 33, 42, 737000), start_station_name='E 1 St & Bowery', start_station_id='5636.13', end_station_name='Spruce St & Nassau St', end_station_id='5137.10', start_lat=40.72486122254819, start_lng=-73.99213135242462, end_lat=40.71146364, end_lng=-74.00552427, member_casual='casual', distance_meters=1869.0514709417869)

### Выведим максимальное, среднее значение, стандартное отклонение и медиан по расстоянию

In [10]:
# Статистики: максимальное, среднее, стандартное отклонение
stats = filtered.agg(
    spark_max('distance_meters').alias('max'),
    avg('distance_meters').alias('mean'),
    stddev('distance_meters').alias('stddev')
).toPandas()

median = filtered.approxQuantile('distance_meters', [0.5], 0.01)[0]
stats['median'] = median

print(stats)


           max         mean       stddev       median
0  21399.46429  1801.268438  1528.764931  1346.075077
